# Lecture 3 · Deploy to AgentCore Runtime

**Prerequisite:** finish `02_build_and_test_locally.ipynb`. The files in `../agent/` must exist — this notebook deploys them.

You have a working agent on your laptop. It only helps you. This lecture makes it a service: an HTTPS endpoint your team, a Slack bot, or a CI job can call, with session isolation, auto-scaling, and full tracing.

### What Runtime gives you that a laptop doesn't

| | Your laptop | AgentCore Runtime |
| --- | --- | --- |
| Reachable by | you | anything that can call HTTPS with SigV4 |
| Concurrency | one conversation | many, each in its own microVM |
| Isolation | none — one process | one session = one microVM, torn down after |
| Max run length | your patience | up to 8 hours per invocation |
| Credentials | your personal AWS keys | a scoped IAM execution role |
| Idle cost | — | zero; billed per second only while running |
| Debugging | `print()` | traces, spans, and logs in CloudWatch |

That fifth row is the important one. Right now your agent can do whatever *you* can do in AWS, which for most learners is everything. In Runtime it gets its own role, and we will scope it down to exactly four EC2 actions on tagged instances.

### What you'll do

1. Write `main.py` — the AgentCore entrypoint
2. Run it locally as an HTTP server and call it with `curl`
3. Install the AgentCore CLI and scaffold a project
4. Deploy, scope the IAM role, invoke
5. Read the traces
6. Note what this course does not cover
7. **Tear it all down**

---
## 1. Where the code goes

The AgentCore CLI expects a specific project layout, and your agent code becomes one folder inside it:

```
AIOpsAgent/                       <- created by `agentcore create`
├── agentcore/
│   ├── agentcore.json            <- your agents and their settings
│   ├── aws-targets.json          <- account + region to deploy to
│   └── cdk/                      <- infrastructure, auto-managed
└── app/
    └── AIOpsAgent/
        ├── main.py               <- entrypoint  (we supply this)
        ├── config.py             <- from notebook 02
        ├── prompts.py            <- from notebook 02
        ├── tools_cloudtrail.py   <- from notebook 02
        ├── tools_ec2.py          <- from notebook 02
        └── pyproject.toml        <- Python dependencies
```

Nothing in `../agent/` needs rewriting for the cloud. We add exactly one file — `main.py` — and copy the folder in.

---
## 2. The entrypoint

This is the only genuinely new code in the lecture, and it is short. `BedrockAgentCoreApp` is a small HTTP server that implements the contract AgentCore Runtime speaks:

- `POST /invocations` — a request for your agent
- `GET /ping` — the health check the platform polls

You never write those routes. You write the function that answers them.

In [ ]:
%%writefile ../agent/main.py
"""AIOps agent - AgentCore Runtime entrypoint.

Run it three ways, same file:

    python main.py                       # local HTTP server on :8080
    agentcore dev                        # local server + browser inspector
    agentcore deploy && agentcore invoke # hosted on AgentCore Runtime
"""

from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext
from strands import Agent
from strands.models import BedrockModel

import config
from prompts import SYSTEM_PROMPT
from tools_cloudtrail import CLOUDTRAIL_TOOLS
from tools_ec2 import EC2_TOOLS

# BedrockAgentCoreApp is the HTTP contract AgentCore Runtime speaks. It gives
# you POST /invocations and GET /ping for free; you supply the logic.
app = BedrockAgentCoreApp()

TOOLS = CLOUDTRAIL_TOOLS + EC2_TOOLS


def build_agent() -> Agent:
    """Construct a fresh agent with its own empty conversation history."""
    model = BedrockModel(
        model_id=config.MODEL_ID,
        region_name=config.AWS_REGION,
        temperature=0.2,          # ops work wants boring, repeatable answers
    )
    return Agent(model=model, system_prompt=SYSTEM_PROMPT, tools=TOOLS)


# AgentCore Runtime gives every session its own isolated microVM and routes a
# session's requests back to the same one, so an in-process dict is enough to
# remember a conversation. Nothing here outlives the session.
_agents: dict[str, Agent] = {}


def get_agent(session_id: str | None) -> Agent:
    key = session_id or "local-session"
    if key not in _agents:
        _agents[key] = build_agent()
    if len(_agents) > 50:                    # bound memory on a long-lived VM
        _agents.pop(next(iter(_agents)))
    return _agents[key]


@app.entrypoint
async def invoke(payload: dict, context: RequestContext):
    """Handle one turn. Yielding strings streams them to the caller."""
    prompt = (payload or {}).get("prompt", "").strip()
    if not prompt:
        yield 'Send a payload like {"prompt": "list my EC2 instances"}.'
        return

    agent = get_agent(context.session_id)
    async for event in agent.stream_async(prompt):
        # Strands emits many event types (tool calls, reasoning, lifecycle).
        # "data" carries user-facing text; forward just that.
        if "data" in event:
            yield event["data"]


if __name__ == "__main__":
    print("AIOps agent starting with config:", config.summary())
    app.run()

### Reading it line by line

**`app = BedrockAgentCoreApp()`**
Creates the server. On your laptop `app.run()` serves it on `localhost:8080`; in the cloud, Runtime serves it. Same object, same code.

**`@app.entrypoint`**
Marks the function that handles one request. Whatever JSON the caller sends arrives as `payload`. By convention that JSON has a `prompt` key — that's why `agentcore invoke --prompt "..."` works — but it is just a dict, and you could put a `ticket_id` or an `account_id` in there too.

**`context: RequestContext`**
The optional second argument, filled in by the platform. Its most useful field is `context.session_id`, which identifies the conversation. It also carries the caller's authorization token and any custom `X-Amzn-Bedrock-AgentCore-Runtime-Custom-*` headers. (If you ever set a session ID yourself, note that Runtime requires **at least 16 characters**.)

**`async def` + `yield`**
Yielding strings instead of returning one turns the response into a stream, so the caller sees words appearing rather than waiting for the whole answer. `agent.stream_async()` emits many event types — tool calls, reasoning steps, lifecycle events — and `if "data" in event` filters that down to just the user-facing text.

> Prefer to keep it simple while learning? A plain synchronous version is equally valid:
> ```python
> @app.entrypoint
> def invoke(payload, context):
>     result = get_agent(context.session_id)(payload["prompt"])
>     return {"result": str(result)}
> ```
> You lose streaming and gain one less concept. Both deploy identically.

**The `_agents` dict**
Runtime gives **each session its own microVM** and routes that session's requests back to the same one. So an ordinary in-process dictionary is enough to remember a conversation — no database, no cache. Two users get two microVMs and cannot see each other's history. When a session ends, the microVM is destroyed and the memory goes with it.

That last sentence is also the limitation: this is memory for *a* conversation, not memory *about a user*. Close the session and the agent forgets you entirely. That is the right trade for this course — section 11 says what you would need to change it.

**`if __name__ == "__main__": app.run()`**


---
## 3. Run the production entrypoint locally

Before any cloud involvement, prove the server works. Two terminals, or the cells below.

**Terminal 1 — start it:**

```bash
cd agent
export AWS_REGION=us-east-1
export AIOPS_DRY_RUN=true
python main.py
```

**Terminal 2 — call it:**

```bash
curl -N -X POST http://localhost:8080/invocations \
  -H 'Content-Type: application/json' \
  -d '{"prompt": "list my EC2 instances"}'
```

`-N` disables curl's buffering so you see the stream arrive. And the health check:

```bash
curl http://localhost:8080/ping
```

If you'd rather not juggle terminals, the next cell does the whole thing from the notebook.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

settings = json.loads(Path("course_settings.json").read_text())
AGENT_DIR = Path("../agent").resolve()

env = {**os.environ,
       "AWS_REGION": settings["region"],
       "AIOPS_MODEL_ID": settings["model_id"],
       "AIOPS_DRY_RUN": "true"}

server = subprocess.Popen([sys.executable, "main.py"], cwd=AGENT_DIR, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print("starting main.py ...")
time.sleep(8)          # give it a moment to bind the port

if server.poll() is not None:
    print("Server exited early. Output:\n")
    print(server.stdout.read())
else:
    print("Server is up on http://localhost:8080")

In [ ]:
import urllib.request

def call_local(prompt: str) -> None:
    request = urllib.request.Request(
        "http://localhost:8080/invocations",
        data=json.dumps({"prompt": prompt}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        for chunk in response:
            print(chunk.decode(errors="replace"), end="")
    print()

# Health check first - this is exactly what Runtime polls.
with urllib.request.urlopen("http://localhost:8080/ping", timeout=10) as r:
    print("GET /ping ->", r.status, r.read().decode())

print("\nPOST /invocations ->")
call_local("List my EC2 instances and tell me which ones I'm allowed to manage.")

In [ ]:
server.terminate()
server.wait(timeout=15)
print("Local server stopped.")

That HTTP exchange is the entire interface between the world and your agent. Runtime adds authentication, isolation, and scale around it — but the contract you just exercised is the contract that runs in production.

---
## 4. Install the AgentCore CLI

The CLI is distributed on npm, so this step needs **Node.js 20 or later**.

```bash
node --version          # must be v20 or higher
npm install -g @aws/agentcore
agentcore --version
```

> **If you previously installed the Python `bedrock-agentcore-starter-toolkit`:** uninstall it. Both packages provide an `agentcore` command and they will shadow each other.
> ```bash
> pip uninstall -y bedrock-agentcore-starter-toolkit
> ```
> The starter toolkit is the legacy CLI; `@aws/agentcore` replaced it. Note this is only about the *CLI* — the `bedrock-agentcore` **Python SDK** you've been importing is current and stays.

The commands you'll use:

| Command | What it does |
| --- | --- |
| `agentcore create` | Scaffold a new project |
| `agentcore dev` | Local server with hot reload + a browser inspector |
| `agentcore deploy` | Provision to AWS (CDK under the hood) |
| `agentcore invoke` | Call the deployed agent |
| `agentcore status` | Show what's deployed, including the execution role |
| `agentcore logs` / `agentcore traces` | Observability |
| `agentcore remove` | Tear things down |

In [ ]:
import shutil, subprocess

for tool in ("node", "npm", "agentcore"):
    path = shutil.which(tool)
    if path:
        version = subprocess.run([tool, "--version"], capture_output=True, text=True).stdout.strip()
        print(f"{tool:<10} {version:<12} {path}")
    else:
        print(f"{tool:<10} NOT FOUND")

---
## 5. Scaffold the project

Run this **in a terminal**, from the repository root — `agentcore create` is an interactive wizard and notebooks handle those badly. The flags below skip the questions:

```bash
cd /path/to/aiops-agent-awsagentcore

agentcore create \
  --name AIOpsAgent \
  --framework Strands \
  --model-provider Bedrock \
  --memory none \
  --build CodeZip
```

What each flag means:

- **`--framework Strands`** — matches what you built in notebook 02. The CLI also supports LangGraph, Google ADK, and OpenAI Agents; your `main.py` would change, the deployment wouldn't.
- **`--model-provider Bedrock`** — reason with Claude on Bedrock. It can also point at Anthropic, OpenAI, or Gemini directly, with keys held in AgentCore Identity.
- **`--memory none`** — no AgentCore Memory. We never turn it on; the per-session dict in `main.py` is all the state this agent has.
- **`--build CodeZip`** — ship a zip of your source rather than building a container. Faster, and no Docker required. Use `--build Container` when you need system packages.



### Copy your agent in

Replace the generated starter code with the files you wrote in notebook 02:

```bash
cp agent/*.py AIOpsAgent/app/AIOpsAgent/
```

Then open `AIOpsAgent/app/AIOpsAgent/pyproject.toml` and make sure the dependencies include what the tools need:

```toml
dependencies = [
    "bedrock-agentcore",
    "strands-agents",
    "boto3",
]
```

### Set the agent's environment variables

Your laptop's shell variables do not travel to the cloud. The deployed agent gets its configuration from `agentcore/agentcore.json`, and there is **no CLI command for this** — you edit the file.

Open `agentcore/agentcore.json` and find your agent under the top-level `runtimes` array. Add an `envVars` entry to it:

```jsonc
{
  "runtimes": [
    {
      "name": "AIOpsAgent",
      "entrypoint": "main.py",
      // ... the fields the wizard generated, left alone ...
      "envVars": [
        { "name": "AIOPS_DRY_RUN",  "value": "true" },
        { "name": "AIOPS_MODEL_ID", "value": "us.amazon.nova-2-lite-v1:0" }
      ]
    }
  ]
}
```

`envVars` is an **array of `{name, value}` objects** — not a `{"KEY": "value"}` map. Names must start with a letter or underscore and contain only letters, digits and underscores. `AWS_REGION` you can leave out; the runtime sets it for you.

Check the file before deploying:

```bash
agentcore validate
```

Then `agentcore deploy` picks the values up. To change one later, edit the file and redeploy.

> **Keep `AIOPS_DRY_RUN=true` for the first deploy.** Prove the plumbing works before you let a hosted agent touch anything. You can flip it to `false` and redeploy in seconds.

> **This is the third place dry run can be set, and they don't talk to each other.** In notebook 02 it's `config.DRY_RUN` in the kernel. Running `main.py` yourself, it's a shell variable. Deployed, it's this file. Setting one has no effect on the others — and none of them is an EC2 tag.

---
## 6. Develop, then deploy

### `agentcore dev`

```bash
cd AIOpsAgent
agentcore dev
```

This creates a virtualenv, installs your dependencies, starts the server with hot reload, and opens the **agent inspector** in your browser — a chat window with a live trace view beside it, so you can watch each tool call as it happens. It runs with *your* AWS credentials, so it behaves like notebook 02.

Use `agentcore dev --no-browser` for a terminal UI instead.

### `agentcore deploy`

```bash
agentcore deploy --dry-run     # show what would change
agentcore deploy               # do it
```

Deploy packages your code into a zip, uses AWS CDK to provision the infrastructure, creates the Runtime endpoint, and wires up CloudWatch logging. **The first run takes several minutes** while CDK bootstraps your account; later ones are much quicker.

```bash
agentcore status
```

`status` prints your endpoint ARN and — the thing you need next — the **execution role name**.

---
## 7. Scope the execution role

This is the step people skip, and it's the one that matters most.

`agentcore deploy` created an execution role with permissions for Runtime itself — invoking Bedrock models, writing logs. It knows nothing about your tools, so **your CloudTrail and EC2 calls will fail with `AccessDenied` until you grant them.**

That failure is a feature. It forces you to state exactly what your agent may do. Here is the policy, from `infra/agentcore-execution-policy.json`:

In [9]:
policy = json.loads(Path("../infra/agentcore-execution-policy.json").read_text())
print(json.dumps(policy, indent=2))

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "CloudTrailReadOnly",
      "Effect": "Allow",
      "Action": [
        "cloudtrail:LookupEvents"
      ],
      "Resource": "*"
    },
    {
      "Sid": "EC2ReadOnly",
      "Effect": "Allow",
      "Action": [
        "ec2:DescribeInstances",
        "ec2:DescribeInstanceStatus",
        "ec2:DescribeTags"
      ],
      "Resource": "*"
    },
    {
      "Sid": "EC2WriteOnlyManagedInstances",
      "Effect": "Allow",
      "Action": [
        "ec2:StartInstances",
        "ec2:StopInstances",
        "ec2:RebootInstances"
      ],
      "Resource": "arn:aws:ec2:*:*:instance/*",
      "Condition": {
        "StringEquals": {
          "aws:ResourceTag/AIOpsManaged": "true"
        }
      }
    }
  ]
}


### Why it's written that way

**`cloudtrail:LookupEvents` on `"*"`** — CloudTrail's lookup API has no resource-level permissions, so `*` is the only option. It is read-only.

**`ec2:Describe*` on `"*"`** — same story: EC2's describe calls don't support resource ARNs. Also read-only.

**The write statement is the interesting one.**

```json
"Action": ["ec2:StartInstances", "ec2:StopInstances", "ec2:RebootInstances"],
"Resource": "arn:aws:ec2:*:*:instance/*",
"Condition": { "StringEquals": { "aws:ResourceTag/AIOpsManaged": "true" } }
```

Three actions. Instances only. **And only instances carrying the tag.** This is the same rule as the `_guard()` function in `tools_ec2.py` — deliberately duplicated, because they fail differently:

- Someone edits `tools_ec2.py` and drops the guard → IAM still refuses.
- Someone gives the role a broader policy → `_guard()` still refuses.
- A clever prompt talks the model past the system prompt → both still refuse.

**Note what is absent.** No `ec2:TerminateInstances`. No `iam:*`. No `s3:*`. An agent should be able to do its job and nothing else. If the agent is ever compromised — a poisoned prompt, a bad tool, a supply-chain issue — this policy is the wall that holds.

**And note the asymmetry:** reads are broad, writes are narrow. That is usually the right shape for an ops agent. It should be able to see everything and change almost nothing.

### Attach it

Get the role name from `agentcore status`, then:

```bash
cd infra
./attach-policy.sh <execution-role-name>
```

Or directly:

```bash
aws iam put-role-policy \
  --role-name <execution-role-name> \
  --policy-name AIOpsAgentToolPermissions \
  --policy-document file://infra/agentcore-execution-policy.json
```

Verify:

```bash
aws iam get-role-policy \
  --role-name <execution-role-name> \
  --policy-name AIOpsAgentToolPermissions
```

> IAM changes take a few seconds to propagate. If the first invoke still says `AccessDenied`, wait ten seconds and try again.

---
## 8. Invoke it

```bash
agentcore invoke --prompt "What has been happening in my AWS account recently?"

agentcore invoke --prompt "List my EC2 instances and say which ones you can manage."

agentcore invoke --prompt "Stop instance i-0abc123def456"
```

That last one exercises the whole stack: the prompt asks for confirmation, `_guard()` checks the tag, dry-run intercepts the call, and IAM stands behind all of it.

To hold a conversation across calls, pass the same session ID (**16 characters minimum**):

```bash
agentcore invoke --session-id my-demo-session-001 --prompt "List my running instances"
agentcore invoke --session-id my-demo-session-001 --prompt "Stop the first one"
```

The second call lands on the same microVM, so `_agents[session_id]` is still there and the agent remembers the inventory. Change the session ID and it starts fresh — that's isolation working.

You can also call the endpoint from any application with the AWS SDK:

```python
import boto3, json
client = boto3.client("bedrock-agentcore", region_name="us-east-1")
response = client.invoke_agent_runtime(
    agentRuntimeArn="<arn from agentcore status>",
    runtimeSessionId="my-demo-session-001",
    payload=json.dumps({"prompt": "list my instances"}).encode(),
)
```

---
## 9. Change a setting and redeploy

That last invoke ran in dry run: the tag check passed, the stop was intercepted, and the tool result said so. Going live is a **configuration** change, and configuration reaches the runtime exactly one way — edit `agentcore.json`, then deploy again.

Open `AIOpsAgent/agentcore/agentcore.json` and flip the one value:

```jsonc
"envVars": [
  { "name": "AIOPS_DRY_RUN",  "value": "false" },   // was "true"
  { "name": "AIOPS_MODEL_ID", "value": "us.amazon.nova-2-lite-v1:0" }
]
```

Then, from the `AIOpsAgent` directory:

```bash
agentcore deploy --dry-run     # preview: one environment variable changes
agentcore deploy               # apply it
```

> **Two different dry runs, one word.** `agentcore deploy --dry-run` is the CLI previewing *infrastructure* changes. `AIOPS_DRY_RUN` is your agent's safety flag. They are unrelated, and the first one is how you inspect a change to the second.

This deploy is quick — your code hasn't changed, so CDK updates the runtime's environment and leaves everything else alone. **Nothing changes in AWS until you run `deploy`.** Editing the file and stopping there is the mistake worth making once and never again.

Now run the same prompt as before:

```bash
agentcore invoke --prompt "Stop instance i-0abc123def456"
```

Different behaviour, same request: no `"dry_run": true` in the tool result, and the instance really stops — provided it carries `AIOpsManaged=true`. If you still get the dry-run message, your edit didn't make it into the deploy; check `git diff agentcore/agentcore.json` and deploy again.

### Why a redeploy is needed at all

`config.py` reads `AIOPS_DRY_RUN` **once, at import**, into `config.DRY_RUN`. Runtime bakes `envVars` into the container's environment when it deploys, so there is no knob to turn on a running endpoint. That isn't a limitation to route around — it's what makes the deployment reproducible: the configuration that is running is the configuration in the file, and the file is in version control.

The same loop covers every setting in `config.py` — model ID, region, the managed tag, the result caps. Edit, `deploy`, invoke.

### Put it back

```bash
# set AIOPS_DRY_RUN back to "true" in agentcore.json, then
agentcore deploy
```

Deleting the entry works too: `config.py` defaults `AIOPS_DRY_RUN` to `true`, so an agent with no such variable is a safe agent.

> **Dry run is off; the tag guard is not.** `_guard()` still refuses any instance without `AIOpsManaged=true`. Two independent layers — you turned one off, and the other is still standing behind it.


---
## 10. Observability

When an agent misbehaves, the question is never "did it crash?" — it's **"why did it decide that?"** Which tool did it pick, with which arguments, and what came back?

```bash
agentcore logs                              # recent output
agentcore logs --since 30m --level error    # just failures
agentcore logs --query "AccessDenied"       # search

agentcore traces list                       # recent invocations
agentcore traces get <trace-id>             # one full trace
```

A trace breaks one invocation into **spans**: the model call, each tool call with its inputs and outputs, and the timing of each. When you run the loop yourself, every one of those steps is yours — and Observability records all of them, so a strange answer from yesterday is something you can go and read rather than guess at.

The same traces appear in the **CloudWatch console → GenAI Observability**, grouped by session so you can follow a whole conversation.



---
## 11. What we are not covering

Two AgentCore services come up in almost every doc page you will read, and you have already typed `--memory none` past one of them. Neither is part of this course, and there is nothing here to run — you should just recognise the names and know which problem each one solves.

**AgentCore Memory** is managed state that outlives a session. Your `_agents` dict remembers a conversation; it cannot remember a *person*. Memory is what stores *"Priya always works in eu-west-1"* or *"this team calls i-0abc 'the billing box'"* across many conversations. Reach for it when your users start repeating themselves.

**AgentCore Gateway** turns things that already exist — a Lambda function, an internal REST API, an OpenAPI spec, an MCP server — into agent tools, with authentication handled for you. Every tool in this course is instead a Python function you wrote, which is the right way to learn and stays right for a small, purpose-built tool set. Gateway is for a large estate you do not want to rewrite.

Both are substantial enough to deserve a course of their own. If you want to read ahead, each has its own chapter in the [AgentCore developer guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/), with runnable examples in [amazon-bedrock-agentcore-samples](https://github.com/awslabs/amazon-bedrock-agentcore-samples). Finish this course first — Runtime, tools, and the safety model are the foundation both of them sit on.


---
## 12. Clean up

**Do this.** An idle Runtime endpoint is cheap but not free, and the CDK stack holds IAM roles and S3 artifacts you don't want lying around.

```bash
cd AIOpsAgent
agentcore remove all
agentcore deploy          # detects the empty state and tears everything down
```

Then remove the inline policy you added by hand — `remove all` doesn't know about it:

```bash
aws iam delete-role-policy \
  --role-name <execution-role-name> \
  --policy-name AIOpsAgentToolPermissions
```

And if you tagged an instance for the demo:

```bash
aws ec2 delete-tags --resources i-0abc123def456 --tags Key=AIOpsManaged
```

Verify nothing is left:

```bash
agentcore status
aws bedrock-agentcore-control list-agent-runtimes --region us-east-1


---
## 13. What it costs

For a demo agent used a few dozen times:

| Item | Rough cost |
| --- | --- |
| Bedrock tokens (Claude Sonnet, ~50 turns with tool calls) | $0.50 – $2.00 |
| AgentCore Runtime (per-second, only while a request is running) | a few cents |
| CloudWatch logs and traces | cents |
| CloudTrail `LookupEvents` | free |
| **Total for this lecture** | **well under $5** |

Tokens dominate, and tool results dominate tokens — every result is re-sent on every subsequent turn. The 5-event cap in `tools_cloudtrail.py` is as much a cost control as a quality one. Switching to a Haiku model cuts the bill several-fold and, for a tool-calling agent this simple, you may not notice the difference.



---
## 14. Exercises

**1. Break IAM on purpose.** Delete the inline policy, invoke the agent, and read the trace. Note that the agent *reports* the failure sensibly instead of crashing — because `_guard()` and boto3 errors both come back as data. Reattach it.

**2. Prove session isolation.** Invoke twice with the same `--session-id` and confirm the agent remembers. Invoke with a different one and confirm it doesn't.

**3. Turn a different knob.** Add `{ "name": "AIOPS_MAX_INSTANCES", "value": "3" }` to `envVars`, redeploy, and ask for ten instances. You get three. Same loop as section 9, different setting - proof that every `os.environ.get` in `config.py` is a deploy-time dial.

**4. Try the harness.** Run `agentcore create` again and pick **Harness** instead of Agent. Build the same agent with config only. Compare honestly: what got easier, and what did you lose visibility into? Knowing when to reach for each is a real engineering judgement, not a matter of taste.



---
## Course recap

Across three notebooks you built and shipped a real agent. Every part of it is something you can open and read:

| Piece | Where it lives | What it does |
| --- | --- | --- |
| Tools | `tools_cloudtrail.py`, `tools_ec2.py` | 9 `@tool` functions — the only things that touch AWS |
| Instructions | `prompts.py` | The standing brief, in git and under review |
| Orchestration | Strands, in your process | The loop, printable and testable |
| Entrypoint | `main.py` | Runs identically on your laptop and in Runtime |
| Safety | prompt + `_guard()` + IAM condition | Three independent layers |
| Deployment | `agentcore deploy` | One command |
| Debugging | Traces and spans | Every tool call, with inputs and outputs |

### The five ideas worth keeping

1. **An agent is a loop.** Model → tool request → your code → result → model. Everything else is plumbing.
2. **A tool is a function, and its docstring is a prompt.** The model picks tools by reading them.
3. **Tool results live in the conversation forever.** Keep them small; it drives both quality and cost.
4. **Prompts are preferences; code and IAM are controls.** Never confuse the two. Build all three.
5. **The same code runs locally and in production.** Build locally, deploy when it works.

### Where to go next

- **AgentCore Memory** — give the agent recall across sessions
- **AgentCore Gateway + MCP** — connect existing APIs without rewriting them
- **Multi-agent** — a read-only investigator handing off to a narrowly-scoped operator
- **Evaluations** — `agentcore add evaluator`, so a prompt change can't quietly make things worse

**[AI Fundamentals for Beginners: Learn LLM, Agentic AI & MCP](https://www.udemy.com/course/ai-fundamentals-for-beginners-learn-llm-agentic-ai-mcp/?referralCode=4D49F0BFDF7A68F7CF22)**